# Sheriffhall Survey — Refactored Analysis Notebook

## Code Review Summary
- Consolidated repeated helper logic into reusable functions.
- Removed global-state column detection patterns by passing DataFrames explicitly.
- Added deterministic color selection for reproducible figures.
- Added defensive validation for required columns and empty datasets.
- Replaced ad-hoc cleaning with centralized preprocessing functions.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

PALETTE = [
    '#8EA4D2', '#6A0136', '#706A58', '#FF8552', '#031A6B',
    '#FF6978', '#DCEED1', '#E0CA3C', '#F24333', '#FE5D9F',
    '#98CE00', '#CFD2B2', '#F2AF29', '#FFD6AF', '#C4F1BE',
    '#F6BD60', '#197BBD', '#F2BAC9', '#8A89C0', '#F4F9E9'
]

def pick_colors(n, palette=PALETTE, seed=42):
    rng = random.Random(seed)
    if n <= len(palette):
        return rng.sample(palette, n)
    expanded = list(palette) * ((n // len(palette)) + 1)
    rng.shuffle(expanded)
    return expanded[:n]

def clean_columns(df):
    result = df.copy()
    result.columns = (
        result.columns
        .str.strip()
        .str.replace('\n', ' ', regex=False)
        .str.replace('\xa0', ' ', regex=False)
    )
    return result

def find_column(df, keywords):
    terms = [keywords] if isinstance(keywords, str) else list(keywords)
    for col in df.columns:
        name = col.lower()
        if any(term.lower() in name for term in terms):
            return col
    return None

def map_trip_purpose(value):
    text = str(value).lower()
    if any(k in text for k in ['work', 'employ', 'job']):
        return 'Work'
    if any(k in text for k in ['leisure', 'social', 'visit', 'friend', 'family']):
        return 'Leisure'
    if 'shop' in text:
        return 'Shopping'
    if any(k in text for k in ['school', 'child', 'education']):
        return 'School'
    if any(k in text for k in ['hospital', 'medical', 'appointment']):
        return 'Medical'
    return 'Other'

def map_mode(value):
    text = str(value).lower().strip()
    if any(k in text for k in ['car', 'vehicle']):
        return 'Car'
    if 'bus' in text:
        return 'Bus'
    if any(k in text for k in ['cycle', 'bike']):
        return 'Cycle'
    if 'walk' in text:
        return 'Walk'
    if 'train' in text:
        return 'Train'
    if 'taxi' in text:
        return 'Taxi'
    return 'Other'

def map_primary_direction(value):
    if pd.isna(value):
        return 'Unknown'
    options = [item.strip().replace(';', '') for item in str(value).split(';') if item.strip()]
    if not options:
        return 'Unknown'
    first = options[0]
    for direction in ['Eastbound', 'Westbound', 'Northbound', 'Southbound']:
        if direction in first:
            return direction
    return 'Other'

def build_behavior_dataset(df):
    frequency_col = find_column(df, ['frequency', 'how often'])
    purpose_col = find_column(df, ['purpose', 'trip purpose'])
    direction_col = find_column(df, ['direction', 'which direction'])
    mode_col = find_column(df, ['preferred mode', 'mode of transport'])
    delay_col = find_column(df, ['peak times', 'how long', 'pass through'])

    data = pd.DataFrame(index=df.index)
    if frequency_col:
        data['Frequency'] = df[frequency_col]
    if purpose_col:
        data['Purpose'] = df[purpose_col].apply(map_trip_purpose)
    if direction_col:
        data['Direction'] = df[direction_col].apply(map_primary_direction)
    if mode_col:
        data['Mode'] = df[mode_col].apply(map_mode)
    if delay_col:
        data['Peak_Delay'] = pd.to_numeric(df[delay_col], errors='coerce')
        data['Delay_Category'] = pd.cut(
            data['Peak_Delay'],
            bins=[0, 10, 20, 30, 1000],
            labels=['<10 min', '10-20 min', '20-30 min', '>30 min']
        )

    return data.dropna()

def plot_correlation_matrix(analysis_df):
    required = ['Frequency', 'Purpose', 'Direction', 'Peak_Delay']
    if not all(col in analysis_df.columns for col in required):
        raise ValueError(f'Missing required columns: {[c for c in required if c not in analysis_df.columns]}')

    encoded = analysis_df.copy()
    encoder = LabelEncoder()
    encoded['Frequency_Enc'] = encoder.fit_transform(encoded['Frequency'])
    encoded['Purpose_Enc'] = encoder.fit_transform(encoded['Purpose'])
    encoded['Direction_Enc'] = encoder.fit_transform(encoded['Direction'])
    encoded['Delay'] = encoded['Peak_Delay']

    matrix = encoded[['Frequency_Enc', 'Purpose_Enc', 'Direction_Enc', 'Delay']].corr()

    plt.figure(figsize=(10, 8))
    im = plt.imshow(matrix.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
    plt.colorbar(im, label='Correlation Coefficient')
    labels = ['Frequency', 'Purpose', 'Direction', 'Delay']
    plt.xticks(range(len(labels)), labels, fontsize=11, fontweight='bold')
    plt.yticks(range(len(labels)), labels, fontsize=11, fontweight='bold')
    plt.title('Travel Behaviour Correlation Matrix', fontsize=13, fontweight='bold')

    for row in range(len(labels)):
        for col in range(len(labels)):
            value = matrix.values[row, col]
            text_color = 'white' if abs(value) >= 0.7 else 'black'
            plt.text(col, row, f'{value:.2f}', ha='center', va='center', color=text_color, fontsize=10)

    plt.tight_layout()
    plt.show()

def plot_mode_count_vs_delay(analysis_df):
    required = ['Mode', 'Peak_Delay']
    if not all(col in analysis_df.columns for col in required):
        raise ValueError(f'Missing required columns: {[c for c in required if c not in analysis_df.columns]}')

    stats = (
        analysis_df
        .groupby('Mode')
        .agg(Avg_Delay=('Peak_Delay', 'mean'), Count=('Peak_Delay', 'size'))
        .round(1)
        .sort_values('Count', ascending=False)
        .head(6)
    )

    colors = pick_colors(len(stats), seed=42)
    fig, ax1 = plt.subplots(figsize=(11, 6))
    bars = ax1.bar(stats.index, stats['Count'], color=colors, edgecolor='black', linewidth=1.0, alpha=0.85)
    ax1.set_xlabel('Mode of Transport', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Number of Respondents', fontsize=11, fontweight='bold')
    ax1.grid(True, alpha=0.25, axis='y')
    ax1.tick_params(axis='x', rotation=35)

    for bar in bars:
        count = int(bar.get_height())
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count), ha='center', va='bottom', fontsize=9)

    ax2 = ax1.twinx()
    ax2.plot(stats.index, stats['Avg_Delay'], color='#c1121f', marker='o', linewidth=2)
    ax2.set_ylabel('Average Peak Delay (minutes)', fontsize=11, fontweight='bold', color='#c1121f')
    ax2.tick_params(axis='y', labelcolor='#c1121f')

    plt.title('Mode Usage vs Average Peak Delay', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Update file_path if needed
file_path = Path('Sheriffhall Roundabout & A720 Bypass_ Community Travel and Congestion Questionnaire.xlsx')
if not file_path.exists():
    raise FileNotFoundError(f'Could not find data file: {file_path.resolve()}')

raw_df = pd.read_excel(file_path)
raw_df = clean_columns(raw_df)
analysis_df = build_behavior_dataset(raw_df)

print(f'Total responses: {len(raw_df)}')
print(f'Complete responses used for analysis: {len(analysis_df)}')
analysis_df.head()

In [ ]:
plot_correlation_matrix(analysis_df)
plot_mode_count_vs_delay(analysis_df)

## Notes
- This refactor is intentionally modular and reproducible.
- Add further thematic plots by creating new functions and reusing `analysis_df`.
- Keep helper logic in one place to avoid duplication across sections.